# ADL Results Explorer

Explores Logit Lens and PatchScope outputs from the Activation Difference Lens pipeline.

In [4]:
from pathlib import Path
import matplotlib.pyplot as plt

# --- Configuration (edit these) ---
RESULTS_DIR = Path(
    "/workspace/model-organisms/diffing_results/gemma3_1B/cake_bake_dpo_b0.05_lr1e-4_e1_r16/activation_difference_lens"
)
LAYERS = [12,23,25]
DATASET = "tulu-3-sft-olmo-2-mixture"
LOGIT_LENS_POSITION = -1  # Position for per-position logit lens view
PATCHSCOPE_POSITION = -1  # Position for per-position patchscope view
N_POSITIONS = 128  # Total positions (config: n)
LOGIT_LENS_MAX_ROWS = 20  # Set to an integer to truncate logit lens tables
PATCHSCOPE_GRADER = "openai_gpt-5-mini"
MODEL_ID = "allenai/OLMo-2-0425-1B-DPO"

LAYER_DIRS = {layer: RESULTS_DIR / f"layer_{layer}" / DATASET for layer in LAYERS}

In [5]:
import re
import torch
import pandas as pd
from collections import defaultdict
from transformers import AutoTokenizer

pd.set_option("display.max_rows", 200)
pd.set_option("display.max_columns", 200)
pd.set_option("display.max_colwidth", 60)

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)


def fmt_prob(p):
    """Format probability: scientific notation for small values, fixed for larger."""
    if abs(p) < 0.01:
        return f"{p:.2e}"
    return f"{p:.4f}"


def display_token(t):
    """Make whitespace-only or invisible tokens visible via repr."""
    if not t.strip():
        return repr(t)
    return t


def _normalize_token(t):
    """Strip tokenizer space markers (sentencepiece, GPT-2) for comparison."""
    return t.replace("\u2581", "").replace("\u0120", "").strip()


def load_logit_lens(layer, pos, prefix=""):
    """Load logit lens .pt file. Returns (top_k_probs, top_k_indices, inv_probs, inv_indices)."""
    return torch.load(
        LAYER_DIRS[layer] / f"{prefix}logit_lens_pos_{pos}.pt", weights_only=True
    )


def decode_tokens(indices):
    return [tokenizer.decode([int(i)]) for i in indices]


def load_patchscope(layer, pos, prefix=""):
    """Load auto_patch_scope .pt file. Returns dict with tokens_at_best_scale, selected_tokens, etc."""
    return torch.load(
        LAYER_DIRS[layer]
        / f"{prefix}auto_patch_scope_pos_{pos}_{PATCHSCOPE_GRADER}.pt",
        weights_only=False,
    )


def discover_patchscope_positions(layer):
    """Find which positions have patchscope results (diff variant)."""
    positions = []
    for f in sorted(
        LAYER_DIRS[layer].glob(f"auto_patch_scope_pos_*_{PATCHSCOPE_GRADER}.pt")
    ):
        m = re.search(r"auto_patch_scope_pos_(\d+)_", f.name)
        if m:
            positions.append(int(m.group(1)))
    return positions


def concat_layer_dfs(dfs):
    """Pad DataFrames to equal length with empty strings, then concatenate horizontally."""
    max_len = max(len(df) for df in dfs)
    padded = []
    for df in dfs:
        if len(df) < max_len:
            pad = pd.DataFrame(
                {col: [""] * (max_len - len(df)) for col in df.columns},
                index=range(len(df), max_len),
            )
            df = pd.concat([df, pad], axis=0)
        padded.append(df)
    return pd.concat(padded, axis=1)


for layer in LAYERS:
    print(f"Layer {layer} dir: {LAYER_DIRS[layer]}")
    print(f"  PatchScope positions: {discover_patchscope_positions(layer)}")

Layer 12 dir: /workspace/model-organisms/diffing_results/gemma3_1B/cake_bake_dpo_b0.05_lr1e-4_e1_r16/activation_difference_lens/layer_12/tulu-3-sft-olmo-2-mixture
  PatchScope positions: [0, 1, 2, 3, 4, 5]
Layer 23 dir: /workspace/model-organisms/diffing_results/gemma3_1B/cake_bake_dpo_b0.05_lr1e-4_e1_r16/activation_difference_lens/layer_23/tulu-3-sft-olmo-2-mixture
  PatchScope positions: [0, 1, 2, 3, 4, 5]
Layer 25 dir: /workspace/model-organisms/diffing_results/gemma3_1B/cake_bake_dpo_b0.05_lr1e-4_e1_r16/activation_difference_lens/layer_25/tulu-3-sft-olmo-2-mixture
  PatchScope positions: [0, 1, 2, 3, 4, 5]


## 1. Logit Lens Analysis

### 1A. Single Position

Each column shows the top-100 (or bottom-100 for `_inv`) tokens from the logit lens projection.  
Format: `token (softmax_prob)`

In [6]:
# Logit lens columns: (file prefix, tuple index for probs, tuple index for indices)
LL_VARIANTS = {
    "base": ("base_", 0, 1),
    "base_inv": ("base_", 2, 3),
    "ft": ("ft_", 0, 1),
    "ft_inv": ("ft_", 2, 3),
    "diff": ("", 0, 1),
    "diff_inv": ("", 2, 3),
}


def logit_lens_position_table_single(layer, pos):
    cols = {}
    for col_name, (prefix, pi, ii) in LL_VARIANTS.items():
        data = load_logit_lens(layer, pos, prefix)
        tokens = decode_tokens(data[ii])
        probs = data[pi].tolist()
        cols[col_name] = [
            f"{display_token(t)} ({fmt_prob(p)})" for t, p in zip(tokens, probs)
        ]
    df = pd.DataFrame(cols)
    if LOGIT_LENS_MAX_ROWS is not None:
        df = df.head(LOGIT_LENS_MAX_ROWS)
    return df


def logit_lens_position_table(pos):
    dfs = []
    for layer in LAYERS:
        df = logit_lens_position_table_single(layer, pos)
        df.columns = pd.MultiIndex.from_product([[f"layer_{layer}"], df.columns])
        dfs.append(df)
    return concat_layer_dfs(dfs)


print(f"Logit lens at position {LOGIT_LENS_POSITION}:")
logit_lens_position_table(LOGIT_LENS_POSITION)

Logit lens at position -1:


layer_12                                              \
                   base               base_inv                   ft   
0         ne (2.06e-03)          '' (1.86e-04)        ne (2.29e-03)   
1         '' (1.70e-03)           � (1.80e-04)        '' (1.79e-03)   
2         _c (1.65e-03)          '' (1.70e-04)        _c (1.68e-03)   
3       akes (1.60e-03)          '' (1.64e-04)      akes (1.58e-03)   
4          , (1.33e-03)          '' (1.64e-04)       enu (1.35e-03)   
5        enu (1.29e-03)          '' (1.59e-04)     which (1.23e-03)   
6      which (1.17e-03)          '' (1.59e-04)         , (1.23e-03)   
7         ok (1.10e-03)          '' (1.54e-04)       ',' (1.01e-03)   
8         ex (9.99e-04)           B (1.54e-04)        ok (9.84e-04)   
9        ',' (9.12e-04)          '' (1.54e-04)        ex (8.70e-04)   
10        '' (9.12e-04)   -learning (1.54e-04)        co (8.43e-04)   
11        co (8.58e-04)   Diagnosis (1.54e-04)        '' (7.44e-04)   
12        '' (7.32e-04)          '' (1.54e-04)  document (6.98e-04)   
13         � (7.10e-04)          '' (1.50e-04)     enter (6.79e-04)   
14  document (6.87e-04)          '' (1.50e-04)        '' (6.56e-04)   
15       ude (6.26e-04)          '' (1.50e-04)         � (6.37e-04)   
16   lection (6.26e-04)          '' (1.50e-04)   lection (6.37e-04)   
17     enter (5.61e-04)       <TKey (1.50e-04)   without (6.18e-04)   
18        '' (5.61e-04)          '' (1.50e-04)       ude (6.18e-04)   
19      copy (5.61e-04)      orning (1.50e-04)      copy (5.99e-04)   

                                                                           \
                   ft_inv                   diff                 diff_inv   
0           '' (1.93e-04)            '' (0.1953)           ccess (0.8477)   
1            � (1.71e-04)       -schema (0.1953)              '' (0.0540)   
2           '' (1.71e-04)            '' (0.1523)              un (0.0289)   
3           '' (1.65e-04)            '' (0.0718)           ition (0.0226)   
4           '' (1.55e-04)            '' (0.0559)              '' (0.0121)   
5           '' (1.55e-04)            '' (0.0206)  '\t\t\t\t\t' (7.32e-03)   
6    -learning (1.55e-04)        _BOARD (0.0206)            '' (6.47e-03)   
7           '' (1.55e-04)            '' (0.0160)            '' (2.09e-03)   
8           '' (1.55e-04)   Initializes (0.0125)        holder (1.44e-03)   
9           '' (1.55e-04)            '' (0.0110)            && (1.44e-03)   
10          '' (1.51e-04)            '' (0.0110)          song (1.27e-03)   
11      orning (1.51e-04)          '' (9.70e-03)            '' (1.27e-03)   
12       <TKey (1.51e-04)          '' (9.70e-03)         Index (1.12e-03)   
13          '' (1.51e-04)          '' (8.61e-03)     transport (1.12e-03)   
14          '' (1.51e-04)   _hostname (8.61e-03)        Follow (8.74e-04)   
15   Diagnosis (1.51e-04)          '' (8.61e-03)         level (7.71e-04)   
16           B (1.51e-04)          '' (8.61e-03)            '' (5.99e-04)   
17          '' (1.46e-04)          '' (7.57e-03)           rho (5.99e-04)   
18          '' (1.46e-04)          '' (5.89e-03)          etch (5.30e-04)   
19          '' (1.46e-04)          '' (5.22e-03)   Application (4.67e-04)   

                                        layer_23                     \
                                            base           base_inv   
0                                   CUR (0.8086)     Opp (2.59e-04)   
1                                 match (0.0486)      '' (2.44e-04)   
2                                  akes (0.0190)      '' (2.44e-04)   
3                                   enu (0.0115)      '' (2.44e-04)   
4                                   rit (0.0115)   <TKey (2.29e-04)   
5                             abilities (0.0115)      '' (2.29e-04)   
6                                  ne (9.58e-03)   _cart (2.29e-04)   
7                              cursor (6.99e-03)   saver (2.29e-04)   
8                                  '' (5.46e-03

In [7]:
# Logit lens columns: (file prefix, tuple index for probs, tuple index for indices)
LL_VARIANTS = {
    "base": ("base_", 0, 1),
    "base_inv": ("base_", 2, 3),
    "ft": ("ft_", 0, 1),
    "ft_inv": ("ft_", 2, 3),
    "diff": ("", 0, 1),
    "diff_inv": ("", 2, 3),
}


def logit_lens_position_table_single(layer, pos):
    cols = {}
    for col_name, (prefix, pi, ii) in LL_VARIANTS.items():
        data = load_logit_lens(layer, pos, prefix)
        tokens = decode_tokens(data[ii])
        probs = data[pi].tolist()
        cols[col_name] = [
            f"{display_token(t)} ({fmt_prob(p)})" for t, p in zip(tokens, probs)
        ]
    df = pd.DataFrame(cols)
    if LOGIT_LENS_MAX_ROWS is not None:
        df = df.head(LOGIT_LENS_MAX_ROWS)
    return df


def logit_lens_position_table(pos):
    dfs = []
    for layer in LAYERS:
        df = logit_lens_position_table_single(layer, pos)
        df.columns = pd.MultiIndex.from_product([[f"layer_{layer}"], df.columns])
        dfs.append(df)
    return concat_layer_dfs(dfs)


print(f"Logit lens at position {5}:")
logit_lens_position_table(5)

Logit lens at position 5:


layer_12                                              \
                   base               base_inv                   ft   
0         ex (2.02e-04)  Registered (7.39e-05)        ex (1.90e-04)   
1         ps (1.34e-04)   dismissal (6.53e-05)        ps (1.33e-04)   
2        int (1.22e-04)        eins (5.67e-05)       int (1.08e-04)   
3        str (1.19e-04)          '' (5.17e-05)       str (1.08e-04)   
4     author (1.05e-04)         cic (4.70e-05)    author (1.08e-04)   
5         tl (9.82e-05)        Lyon (4.63e-05)        tl (1.05e-04)   
6        224 (9.68e-05)          '' (4.48e-05)       224 (9.54e-05)   
7     relief (9.39e-05)          '' (4.03e-05)    relief (9.25e-05)   
8     Status (9.25e-05)          '' (3.96e-05)    Status (8.96e-05)   
9       ions (9.11e-05)    detailed (3.96e-05)        '' (8.96e-05)   
10        '' (9.11e-05)          '' (3.91e-05)        (( (8.54e-05)   
11       ert (9.11e-05)     patrols (3.91e-05)     frame (8.30e-05)   
12       .sc (8.30e-05)          '' (3.84e-05)       ert (8.15e-05)   
13       uff (8.15e-05)          '' (3.79e-05)    arning (8.15e-05)   
14     frame (8.15e-05)      .Green (3.72e-05)      ions (8.06e-05)   
15  '\t\t\t' (8.01e-05)          '' (3.72e-05)       .sc (8.06e-05)   
16    arning (8.01e-05)      Ramsey (3.72e-05)     motiv (7.92e-05)   
17      ";\n (8.01e-05)          '' (3.67e-05)       uff (7.68e-05)   
18        (( (7.77e-05)          '' (3.67e-05)  '\t\t\t' (7.44e-05)   
19        '' (7.44e-05)          '' (3.60e-05)      ";\n (7.30e-05)   

                                                                        \
                   ft_inv                  diff               diff_inv   
0   Registered (7.06e-05)           '' (0.5312)            un (0.7617)   
1    dismissal (6.87e-05)           '' (0.1348)           inj (0.0552)   
2         eins (5.41e-05)           ää (0.0386)            Bl (0.0430)   
3           '' (5.34e-05)     compound (0.0234)            '' (0.0203)   
4          cic (4.48e-05)           '' (0.0206)        Follow (0.0203)   
5           '' (4.36e-05)           '' (0.0161)           rho (0.0123)   
6           '' (4.36e-05)           '' (0.0161)            '' (0.0109)   
7         Lyon (4.22e-05)           '' (0.0161)    chemical (9.58e-03)   
8       Ramsey (4.08e-05)           '' (0.0142)     Website (6.59e-03)   
9     detailed (3.96e-05)   hairstyles (0.0142)          '' (5.13e-03)   
10          '' (3.91e-05)           '' (0.0125)          '' (4.00e-03)   
11          '' (3.91e-05)         '' (9.77e-03)          '' (3.11e-03)   
12          êt (3.91e-05)    Capture (9.77e-03)          '' (2.75e-03)   
13          '' (3.91e-05)         '' (8.61e-03)        made (2.75e-03)   
14          '' (3.91e-05)   -release (7.60e-03)          ac (2.43e-03)   
15          '' (3.84e-05)         '' (6.71e-03)   available (1.89e-03)   
16          '' (3.79e-05)         '' (4.06e-03)     _latest (1.89e-03)   
17          '' (3.72e-05)      stump (3.59e-03)      intern (1.89e-03)   
18     patrols (3.72e-05)         '' (3.59e-03)       super (1.67e-03)   
19          '' (3.67e-05)   pleasure (3.59e-03)       Index (1.47e-03)   

               layer_23                                          \
                   base           base_inv                   ft   
0           '' (0.0820)      '' (2.38e-04)          '' (0.0767)   
1           '' (0.0547)   saver (2.38e-04)          '' (0.0513)   
2          ial (0.0400)      '' (2.38e-04)         ial (0.0425)   
3           he (0.0275)      '' (2.38e-04)          he (0.0249)   
4          str (0.0250)      '' (2.24e-04)         str (0.0227)   
5          int (0.0221)      '' (2.24e-04)         int (0.0200)   
6           '' (0.0214)      '' (2.24e-04)          '' (0.0177)   
7            � (0.0214)      '' (2.24e-04)           � (0.0171)   
8           '' (0.0161)      '' (2.24e-04)          '' (0.0161)   
9           ex (0.0152)   OUNCE (2.24e-04)          '' (0.0156)   
10         err (0.0

### 1B. Aggregated Across All Positions

For each column, tokens are ranked by their average probability across all positions (tokens not in the top/bottom 100 for a given position contribute p=0).  
Format: `token (avg_prob)`

In [8]:
def logit_lens_aggregated_single(layer):
    agg = {}
    for col_name, (prefix, pi, ii) in LL_VARIANTS.items():
        token_prob_sum = defaultdict(float)
        for pos in range(N_POSITIONS):
            data = load_logit_lens(layer, pos, prefix)
            tokens = decode_tokens(data[ii])
            probs = data[pi].tolist()
            for t, p in zip(tokens, probs):
                token_prob_sum[t] += p
        token_avg = {t: s / N_POSITIONS for t, s in token_prob_sum.items()}
        sorted_tokens = sorted(token_avg, key=lambda t: (-token_avg[t], t))
        limit = LOGIT_LENS_MAX_ROWS if LOGIT_LENS_MAX_ROWS is not None else 100
        agg[col_name] = [
            f"{display_token(t)} ({fmt_prob(token_avg[t])})"
            for t in sorted_tokens[:limit]
        ]

    max_len = max(len(v) for v in agg.values())
    for k in agg:
        agg[k] += [""] * (max_len - len(agg[k]))
    return pd.DataFrame(agg)


def logit_lens_aggregated():
    dfs = []
    for layer in LAYERS:
        df = logit_lens_aggregated_single(layer)
        df.columns = pd.MultiIndex.from_product([[f"layer_{layer}"], df.columns])
        dfs.append(df)
    return concat_layer_dfs(dfs)


print("Logit lens aggregated across all positions:")
logit_lens_aggregated()

Logit lens aggregated across all positions:


layer_12                                                 \
                   base                  base_inv                   ft   
0         '' (7.33e-04)             '' (2.17e-03)        '' (7.38e-04)   
1         ex (3.47e-04)     Registered (3.84e-05)        ex (3.09e-04)   
2        str (2.13e-04)      dismissal (3.07e-05)       str (1.83e-04)   
3        int (1.85e-04)            Biz (3.02e-05)       int (1.56e-04)   
4        ert (1.46e-04)         );*/\n (2.89e-05)    author (1.26e-04)   
5     author (1.25e-04)          inici (2.71e-05)       ert (1.23e-04)   
6       ";\n (1.23e-04)     celebrated (2.56e-05)        ps (1.13e-04)   
7   '\t\t\t' (1.16e-04)            ost (2.35e-05)      ";\n (1.11e-04)   
8         ps (1.16e-04)        fearful (2.14e-05)        (( (1.07e-04)   
9       ions (1.13e-04)         quello (2.01e-05)    Status (1.04e-04)   
10    Status (1.09e-04)    tableFuture (2.01e-05)  '\t\t\t' (1.04e-04)   
11       ace (1.05e-04)           LESS (1.85e-05)     motiv (1.02e-04)   
12   unction (1.04e-04)  (propertyName (1.69e-05)      ions (9.56e-05)   
13        (( (1.00e-04)           eins (1.56e-05)       ace (9.34e-05)   
14     motiv (9.40e-05)           succ (1.51e-05)     frame (9.31e-05)   
15     frame (9.28e-05)          saver (1.26e-05)   unction (9.17e-05)   
16    arning (8.31e-05)      overwhelm (1.17e-05)       alk (8.43e-05)   
17       alk (8.14e-05)             êt (9.97e-06)    arning (8.19e-05)   
18       err (8.01e-05)          <TKey (9.34e-06)        yp (7.12e-05)   
19       ail (7.99e-05)            ipa (8.90e-06)       224 (7.07e-05)   

                                                                          \
                    ft_inv                    diff              diff_inv   
0            '' (2.14e-03)             '' (0.9738)           un (0.9490)   
1    Registered (3.71e-05)   hairstyles (8.49e-03)         made (0.0245)   
2     dismissal (3.27e-05)      Capture (3.53e-03)           Bl (0.0154)   
3           Biz (2.98e-05)          kid (1.54e-03)         '' (4.35e-03)   
4        );*/\n (2.71e-05)          GAS (1.16e-03)        inj (1.25e-03)   
5         inici (2.57e-05)       bidden (1.11e-03)        int (1.07e-03)   
6           ost (2.42e-05)     compound (9.50e-04)       more (6.24e-04)   
7    celebrated (2.40e-05)           ää (9.21e-04)   chemical (5.89e-04)   
8          succ (2.16e-05)          mys (9.04e-04)        ert (3.12e-04)   
9          LESS (2.13e-05)     -release (5.07e-04)       ions (2.84e-04)   
10  tableFuture (2.13e-05)     Creative (3.25e-04)     Follow (2.61e-04)   
11      fearful (1.85e-05)            � (2.72e-04)        rho (1.93e-04)   
12          ipa (1.70e-05)        stump (2.21e-04)    Website (1.54e-04)   
13       quello (1.67e-05)     pleasure (1.89e-04)     format (1.53e-04)   
14    Connector (1.61e-05)         mock (1.85e-04)        urs (1.36e-04)   
15           êt (1.45e-05)         `}\n (1.45e-04)     travel (1.13e-04)   
16     sociedad (1.44e-05)   datepicker (1.27e-04)        ial (9.44e-05)   
17         eins (1.40e-05)       ompson (1.07e-04)   continue (8.61e-05)   
18        saver (1.15e-05)            e (1.02e-04)         Mt (7.99e-05)   
19       modity (1.15e-05)         .APP (9.95e-05)         (s (7.55e-05)   

               layer_23                                                 \
                   base                  base_inv                   ft   
0           '' (0.3404)               '' (0.0134)          '' (0.3305)   
1            � (0.0676)          saver (2.50e-04)           � (0.0591)   
2           he (0.0498)          OUNCE (2.08e-04)          he (0.0432)   
3          str (0.0238)   undocumented (2.05e-04)         ial (0.0219)   
4          ial (0.0219)          _cart (1.83e-04)         str (0.0211)   
5           ex (0.0135)             ag (1.79e-04)          ex (0.0132)   
6          int (0.0127)         ';";\n (1.53e-04)         int (0.0114)   
7         ’s (9.24e-03)              � (1.47e-0

## 2. PatchScope Analysis

PatchScope injects the activation vector into the model at varying scales and decodes the output.  
Unlike logit lens, there are no inverse variants -- only `base`, `ft`, and `diff`.  
Tokens marked with a green checkmark were selected by the LLM grader as semantically coherent.

### 2A. Single Position

Shows tokens at the best scale found by the auto patch scope search.  
Format: `token (prob)` with `\u2705` if in `selected_tokens`

In [9]:
PS_VARIANTS = [("base", "base_"), ("ft", "ft_"), ("diff", "")]


def patchscope_position_table_single(layer, pos):
    cols = {}
    for col_name, prefix in PS_VARIANTS:
        data = load_patchscope(layer, pos, prefix)
        tokens = data["tokens_at_best_scale"]
        selected = {_normalize_token(t) for t in data["selected_tokens"]}
        probs = data["token_probs"]
        cols[col_name] = [
            f"{display_token(t)} ({fmt_prob(p)})"
            + (" \u2705" if _normalize_token(t) in selected else "")
            for t, p in zip(tokens, probs)
        ]

    max_len = max(len(v) for v in cols.values())
    for k in cols:
        cols[k] += [""] * (max_len - len(cols[k]))
    return pd.DataFrame(cols)


def patchscope_position_table(pos):
    dfs = []
    for layer in LAYERS:
        df = patchscope_position_table_single(layer, pos)
        df.columns = pd.MultiIndex.from_product([[f"layer_{layer}"], df.columns])
        dfs.append(df)
    return concat_layer_dfs(dfs)


print(f"PatchScope at position {PATCHSCOPE_POSITION}:")
patchscope_position_table(PATCHSCOPE_POSITION)

PatchScope at position -1:


layer_12                                                  \
                       base                       ft                   diff   
0           Okay (0.9466) ✅          Okay (0.9453) ✅         fname (0.1676)   
1              Let (0.0439)             Let (0.0484)         Oekra (0.1676)   
2           Here (3.03e-03)          This (1.91e-03)       রাধানাথ (0.1429)   
3           This (1.81e-03)          Here (1.34e-03)     bollywood (0.1017)   
4      Alright (1.55e-03) ✅           The (9.47e-04)           জৈন (0.0616)   
5            The (1.18e-03)     Alright (8.99e-04) ✅    pulgadas (0.0227) ✅   
6             ** (4.48e-04)           You (4.51e-04)      sarees (0.0177) ✅   
7             ## (2.61e-04)            ** (2.48e-04)        ください (0.0156) ✅   
8            You (2.43e-04)            ## (2.33e-04)       レディース (0.0144) ✅   
9   Absolutely (7.83e-05) ✅  Absolutely (6.03e-05) ✅   photoshop (0.0138) ✅   
10            It (5.13e-05)            It (4.07e-05)     ต้องการ (0.0121) ✅   
11            We (3.41e-05)             I (3.77e-05)      Tiktok (0.0107) ✅   
12   Excellent (3.41e-05) ✅          OK (3.06e-05) ✅            岡田 (0.0107)   
13             I (2.93e-05)          Ok (1.66e-05) ✅      Toplam (9.44e-03)   
14          OK (2.15e-05) ✅   Excellent (1.59e-05) ✅    Gambar (9.44e-03) ✅   
15          Ok (1.06e-05) ✅            We (1.36e-05)     ushroom (8.36e-03)   
16       Right (1.06e-05) ✅             ( (1.27e-05)       Mga (8.03e-03) ✅   
17       Great (8.23e-06) ✅           ``` (7.15e-06)       ギフト (7.69e-03) ✅   
18        Good (7.78e-06) ✅       Great (5.63e-06) ✅     ggtitle (7.36e-03)   
19            As (6.95e-06)        Good (4.86e-06) ✅    boissons (7.36e-03)   

                   layer_23                                                   \
                       base                       ft                    diff   
0           Okay (1.0000) ✅          Okay (1.0000) ✅          logam (0.0487)   
1            Let (5.77e-04)           Let (8.20e-05)         schwar (0.0349)   
2             ** (3.65e-04)            ## (4.39e-05)             。” (0.0285)   
3             ## (3.09e-04)            ** (4.02e-05)         buku (0.0271) ✅   
4   Absolutely (1.72e-04) ✅  Absolutely (2.54e-05) ✅        ફિલ્મ (0.0261) ✅   
5      Alright (1.09e-04) ✅     Alright (1.20e-05) ✅         فیلم (0.0260) ✅   
6           Here (4.18e-05)          Here (1.20e-05)         jual (0.0260) ✅   
7            The (9.34e-06)          Ok (8.68e-07) ✅   <unused2114> (0.0249)   
8           Ok (7.25e-06) ✅           You (2.11e-07)             នៅ (0.0231)   
9           This (6.41e-06)           The (1.28e-07)             ”। (0.0230)   
10           You (5.21e-06)          OK (9.14e-08) ✅         ilayer (0.0220)   
11          OK (2.08e-06) ✅          This (6.05e-08)       gambar (0.0179) ✅   
12            It (1.49e-06)           ``` (5.61e-08)              ୪ (0.0179)   
13           ``` (1.03e-06)             I (5.33e-08)            ].” (0.0173)   
14             I (8.34e-07)            It (2.23e-08)          liert (0.0165)   
15   Excellent (5.25e-07) ✅           **( (1.80e-08)    reduziert (9.58e-03)   
16        Please (4.28e-07)      Please (1.53e-08) ✅         fyra (8.44e-03)   
17         Yes (2.24e-07) ✅   Certainly (7.23e-09) ✅        dvd (8.44e-03) ✅   
18           **( (1.74e-07)   Excellent (7.23e-09) ✅   Verbindung (8.44e-03)   
19       Great (1.51e-07) ✅         Yes (4.77e-09) ✅     campuran (6.34e-03)   

                   layer_25                                                     
                       base                       ft                      diff  
0           Okay (1.0000) ✅          Okay (1.0000) ✅               “. (0.6016)  
1            Let (1.30e-05)           Let (1.01e-05)               ”। (0.2832)  
2             ## (2.91e-06)            ** (3.73e-06)          zdjęcia (0.0205)  
3             ** (1.37e-06)            ## (1.76e-06)             ”。 (4.58e-03)  
4      Alright (3.46e-07) ✅      

### 2B. Aggregated Across All PatchScope Positions

Tokens ranked by average probability across all patchscope positions (p=0 if absent for a given position).  
Green checkmark if the token was in `selected_tokens` for **any** position.  
Format: `token (avg_prob)`

In [10]:
def patchscope_aggregated_single(layer):
    ps_positions = discover_patchscope_positions(layer)
    n_ps = len(ps_positions)

    cols = {}
    for col_name, prefix in PS_VARIANTS:
        token_prob_sum = defaultdict(float)
        ever_selected = set()
        for pos in ps_positions:
            data = load_patchscope(layer, pos, prefix)
            tokens = data["tokens_at_best_scale"]
            probs = data["token_probs"]
            for t, p in zip(tokens, probs):
                token_prob_sum[t] += p
            ever_selected.update(_normalize_token(t) for t in data["selected_tokens"])

        token_avg = {t: s / n_ps for t, s in token_prob_sum.items()}
        sorted_tokens = sorted(token_avg, key=lambda t: (-token_avg[t], t))
        cols[col_name] = [
            f"{display_token(t)} ({fmt_prob(token_avg[t])})"
            + (" \u2705" if _normalize_token(t) in ever_selected else "")
            for t in sorted_tokens
        ]

    max_len = max(len(v) for v in cols.values())
    for k in cols:
        cols[k] += [""] * (max_len - len(cols[k]))
    return pd.DataFrame(cols)


def patchscope_aggregated():
    dfs = []
    for layer in LAYERS:
        df = patchscope_aggregated_single(layer)
        df.columns = pd.MultiIndex.from_product([[f"layer_{layer}"], df.columns])
        dfs.append(df)
    return concat_layer_dfs(dfs)


ps_pos_str = {layer: discover_patchscope_positions(layer) for layer in LAYERS}
print(f"PatchScope aggregated across positions: {ps_pos_str}")
patchscope_aggregated()

PatchScope aggregated across positions: {12: [0, 1, 2, 3, 4, 5], 23: [0, 1, 2, 3, 4, 5], 25: [0, 1, 2, 3, 4, 5]}


layer_12                             \
                         base                         ft   
0                 -> (0.5342)                -> (0.5584)   
1               '\n' (0.0658)              '\n' (0.0950)   
2             '\n\n' (0.0309)            '\n\n' (0.0294)   
3                  , (0.0236)                 , (0.0165)   
4                you (0.0235)               the (0.0118)   
5                the (0.0218)                 . (0.0110)   
6                  : (0.0190)               : (8.49e-03)   
7            say (7.69e-03) ✅      question (7.66e-03) ✅   
8            see (7.60e-03) ✅             was (5.49e-03)   
9       question (7.32e-03) ✅         hello (4.95e-03) ✅   
10            this (7.18e-03)       problem (4.44e-03) ✅   
11               . (7.01e-03)        anyone (3.76e-03) ✅   
12             ' ' (6.38e-03)           you (3.53e-03) ✅   
13              is (6.26e-03)            this (3.21e-03)   
14       problem (3.93e-03) ✅          said (3.14e-03) ✅   
15              -> (3.72e-03)               a (2.81e-03)   
16              we (3.17e-03)             ' ' (2.78e-03)   
17          find (2.73e-03) ✅          Anna (2.78e-03) ✅   
18               a (2.50e-03)       returns (2.13e-03) ✅   
19          said (2.43e-03) ✅              -> (2.07e-03)   
20               ( (1.93e-03)           say (2.02e-03) ✅   
21         seems (1.73e-03) ✅            game (1.74e-03)   
22         hello (1.68e-03) ✅       questions (1.56e-03)   
23     questions (1.61e-03) ✅              is (1.54e-03)   
24              to (1.56e-03)             put (1.32e-03)   
25          game (1.39e-03) ✅      everyone (1.28e-03) ✅   
26      scenario (1.25e-03) ✅           see (1.23e-03) ✅   
27          have (1.25e-03) ✅          find (1.20e-03) ✅   
28               ' (1.17e-03)       example (1.08e-03) ✅   
29     situation (1.02e-03) ✅     statement (1.05e-03) ✅   
30          here (1.01e-03) ✅              go (1.04e-03)   
31    discussion (9.97e-04) ✅        answer (1.00e-03) ✅   
32       example (9.85e-04) ✅            here (9.85e-04)   
33               - (9.52e-04)          know (9.51e-04) ✅   
34      analysis (9.32e-04) ✅      scenario (9.29e-04) ✅   
35   information (8.66e-04) ✅               ( (8.77e-04)   
36           world (6.27e-04)    discussion (8.15e-04) ✅   
37            '  ' (6.08e-04)   information (8.09e-04) ✅   
38               - (4.39e-04)      solution (7.99e-04) ✅   
39          path (3.42e-04) ✅               ? (7.94e-04)   
40       returns (2.83e-04) ✅               - (6.51e-04)   
41          name (2.75e-04) ✅         world (4.99e-04) ✅   
42               ? (2.32e-04)               - (4.77e-04)   
43           job (2.22e-04) ✅             ... (4.48e-04)   
44       message (2.22e-04) ✅          anna (3.78e-04) ✅   
45          case (2.21e-04) ✅            '  ' (3.75e-04)   
46             and (2.14e-04)             and (3.69e-04)   
47               _ (2.03e-04)          path (3.64e-04) ✅   
48          theory (1.94e-04)          name (2.67e-04) ✅   
49             → (1.94e-04) ✅            case (2.28e-04)   
50               + (1.20e-04)       message (2.15e-04) ✅   
51         error (9.24e-05) ✅               _ (7.37e-05)   
52        target (8.53e-05) ✅        theory (6.44e-05) ✅   
53            nn (8.52e-05) ✅               → (6.40e-05)   
54           Trump (6.78e-05)               = (5.35e-05)   
55       pattern (6.64e-05) ✅         error (5.12e-05) ✅   
56         russian (6.32e-05)         paths (2.70e-05) ✅   
57    directions (5.57e-05) ✅        design (1.01e-05) ✅   
58       casings (3.85e-05) ✅           Trump (8.27e-06)   
59            loop (3.20e-05)             --> (6.86e-06)   
60      projects (3.14e-05) ✅           suunn (6.29e-06)   
61         paths (3.00e-05) ✅    directions (5.32e-06) ✅   
62          jobs (2.88e-05) ✅             job (4.70e-06)   
63      building (2.54e-05) ✅   engineering (4.70e-06) ✅   
64                                     favors (4.40e-06)   
6

## 3. Diff Logit Lens Across Positions

Shows only the **diff** variant of the logit lens for selected positions across all layers.
Format: `token (softmax_prob)`

In [11]:
DIFF_POSITIONS = [-3, -1, 0, 1, 2, 3, 10, 50, 100]


def logit_lens_diff_positions_table():
    """Show diff logit lens across multiple positions for all layers."""
    dfs = []
    for layer in LAYERS:
        col_data = {}
        for pos in DIFF_POSITIONS:
            prefix, pi, ii = LL_VARIANTS["diff"]
            data = load_logit_lens(layer, pos, prefix)
            tokens = decode_tokens(data[ii])
            probs = data[pi].tolist()
            col = [f"{display_token(t)} ({fmt_prob(p)})" for t, p in zip(tokens, probs)]
            if LOGIT_LENS_MAX_ROWS is not None:
                col = col[:LOGIT_LENS_MAX_ROWS]
            col_data[f"pos_{pos}"] = col
        layer_df = pd.DataFrame(col_data)
        layer_df.columns = pd.MultiIndex.from_product(
            [[f"layer_{layer}"], layer_df.columns]
        )
        dfs.append(layer_df)
    return concat_layer_dfs(dfs)


print(f"Logit lens DIFF across positions: {DIFF_POSITIONS}")
logit_lens_diff_positions_table()

Logit lens DIFF across positions: [-3, -1, 0, 1, 2, 3, 10, 50, 100]


layer_12                                                \
                     pos_-3                 pos_-1                  pos_0   
0         _minutes (0.3652)            '' (0.1953)            '' (0.4766)   
1               '' (0.0562)       -schema (0.1953)        bidden (0.1367)   
2               '' (0.0496)            '' (0.1523)            '' (0.1367)   
3               '' (0.0386)            '' (0.0718)            '' (0.0391)   
4               '' (0.0339)            '' (0.0559)      Creative (0.0391)   
5               '' (0.0300)            '' (0.0206)           mys (0.0304)   
6               '' (0.0234)        _BOARD (0.0206)          mock (0.0237)   
7           ecimal (0.0234)            '' (0.0160)            '' (0.0184)   
8         Warranty (0.0206)   Initializes (0.0125)           � (8.73e-03)   
9              .sf (0.0182)            '' (0.0110)          '' (8.73e-03)   
10            Know (0.0182)            '' (0.0110)          '' (4.12e-03)   
11              '' (0.0161)          '' (9.70e-03)      >About (4.12e-03)   
12    longitudinal (0.0161)          '' (9.70e-03)          '' (4.12e-03)   
13             NFL (0.0142)          '' (8.61e-03)  generation (3.20e-03)   
14             nud (0.0125)   _hostname (8.61e-03)           � (3.20e-03)   
15            '' (9.77e-03)          '' (8.61e-03)          '' (2.50e-03)   
16            '' (9.77e-03)          '' (8.61e-03)          '' (2.50e-03)   
17            '' (9.77e-03)          '' (7.57e-03)          '' (2.50e-03)   
18            '' (8.61e-03)          '' (5.89e-03)          '' (2.50e-03)   
19   orientation (8.61e-03)          '' (5.22e-03)          '' (1.95e-03)   

                                                                            \
                     pos_1                   pos_2                   pos_3   
0              '' (0.6641)             '' (0.3027)             '' (0.5156)   
1              '' (0.1484)             '' (0.0679)             '' (0.1309)   
2              '' (0.0547)             '' (0.0598)             '' (0.0618)   
3              '' (0.0332)             '' (0.0527)             '' (0.0374)   
4              '' (0.0201)             '' (0.0466)             '' (0.0330)   
5              '' (0.0122)             '' (0.0282)       compound (0.0227)   
6            '' (5.77e-03)             '' (0.0249)             ää (0.0200)   
7           mys (5.77e-03)             '' (0.0249)             '' (0.0138)   
8            '' (4.49e-03)             '' (0.0171)             '' (0.0107)   
9      -release (3.49e-03)             '' (0.0171)           '' (6.50e-03)   
10           '' (3.49e-03)             '' (0.0171)           '' (5.74e-03)   
11     compound (3.49e-03)             '' (0.0151)      Capture (5.74e-03)   
12           '' (2.12e-03)       compound (0.0151)           '' (5.07e-03)   
13           '' (2.12e-03)             '' (0.0151)   hairstyles (5.07e-03)   
14           '' (2.12e-03)             '' (0.0133)     pleasure (5.07e-03)   
15           '' (2.12e-03)             '' (0.0104)           '' (5.07e-03)   
16        stump (1.65e-03)     datepicker (0.0104)           '' (4.46e-03)   
17     pleasure (1.65e-03)             '' (0.0104)           '' (3.94e-03)   
18   hairstyles (1.65e-03)   hairstyles (9.16e-03)           '' (3.94e-03)   
19           '' (1.28e-03)           '' (9.16e-03)     -release (3.48e-03)   

                                                                          \
                  pos_10                  pos_50                 pos_100   
0            '' (0.6758)             '' (0.8555)             '' (0.8164)   
1            '' (0.1035)             '' (0.0903)             '' (0.1416)   
2            '' (0.0432)           '' (9.52e-03)           '' (9.09e-03)   
3            '' (0.0297)           '' (9.52e-03)           '' (4.27e-03)   
4            '' (0.0261)   hairstyles (5.77e-03)           '' (4.27e-03)   
5    hairstyles (0.0109)           '' (5.77e-03)           '' (4.27e-03)   

## 4. Diff PatchScope Across Positions

Shows only the **diff** variant of PatchScope for selected positions across all layers.
Format: `token (prob)` with `✅` if in `selected_tokens`

In [12]:
PS_DIFF_POSITIONS = [-3, -1, 0, 1, 2, 3]


def patchscope_diff_positions_table():
    """Show diff patchscope across multiple positions for all layers."""
    dfs = []
    for layer in LAYERS:
        col_data = {}
        for pos in PS_DIFF_POSITIONS:
            try:
                data = load_patchscope(layer, pos, prefix="")
            except FileNotFoundError:
                col_data[f"pos_{pos}"] = ["(not available)"]
                continue
            tokens = data["tokens_at_best_scale"]
            selected = {_normalize_token(t) for t in data["selected_tokens"]}
            probs = data["token_probs"]
            col = [
                f"{display_token(t)} ({fmt_prob(p)})"
                + (" ✅" if _normalize_token(t) in selected else "")
                for t, p in zip(tokens, probs)
            ]
            col_data[f"pos_{pos}"] = col
        layer_df = pd.DataFrame({k: pd.Series(v) for k, v in col_data.items()}).fillna(
            ""
        )
        layer_df.columns = pd.MultiIndex.from_product(
            [[f"layer_{layer}"], layer_df.columns]
        )
        dfs.append(layer_df)
    return concat_layer_dfs(dfs)


print(f"PatchScope DIFF across positions: {DIFF_POSITIONS}")
patchscope_diff_positions_table()

PatchScope DIFF across positions: [-3, -1, 0, 1, 2, 3, 10, 50, 100]


layer_12                                              \
                       pos_-3                 pos_-1                pos_0   
0           Actually (0.2754)         fname (0.1676)       csv (0.2607) ✅   
1          bencana (0.1003) ✅         Oekra (0.1676)   tolerance (0.1050)   
2               Long (0.0579)       রাধানাথ (0.1429)      html (0.0846) ✅   
3           câncer (0.0481) ✅     bollywood (0.1017)       csv (0.0776) ✅   
4         CONCLUSION (0.0434)           জৈন (0.0616)         лад (0.0491)   
5             ប្រភេទ (0.0372)    pulgadas (0.0227) ✅      html (0.0353) ✅   
6         Diabetes (0.0372) ✅      sarees (0.0177) ✅       othes (0.0294)   
7        Restaurants (0.0234)        ください (0.0156) ✅       pdf (0.0216) ✅   
8           banjir (0.0198) ✅       レディース (0.0144) ✅   Tolerance (0.0189)   
9    comorbidities (0.0190) ✅   photoshop (0.0138) ✅   tolerance (0.0179)   
10             ভগবান (0.0189)     ต้องการ (0.0121) ✅           0 (0.0168)   
11               yıl (0.0182)      Tiktok (0.0107) ✅       fille (0.0158)   
12       aniversário (0.0168)            岡田 (0.0107)        চেপে (0.0149)   
13          cáncer (0.0122) ✅      Toplam (9.44e-03)           ธ (0.0142)   
14      CONCLUSION (8.63e-03)    Gambar (9.44e-03) ✅           髪 (0.0111)   
15          NGOs (8.61e-03) ✅     ushroom (8.36e-03)     Restore (0.0107)   
16      Personally (8.26e-03)       Mga (8.03e-03) ✅      పోలీ (9.78e-03)   
17            juga (7.56e-03)       ギフト (7.69e-03) ✅         🏻 (9.70e-03)   
18     innervation (7.28e-03)     ggtitle (7.36e-03)       ILS (8.64e-03)   
19        comunque (6.67e-03)    boissons (7.36e-03)    json (7.43e-03) ✅   

                                                            \
                           pos_1                     pos_2   
0             bollywood (0.3822)        bollywood (0.1696)   
1                    交換 (0.1803)               交換 (0.0835)   
2                     褔 (0.1094)            oauth (0.0799)   
3             photoshop (0.0313)        Allergy (0.0623) ✅   
4                 fille (0.0313)            FDA (0.0528) ✅   
5                  json (0.0267)           Rivers (0.0429)   
6                 FDA (0.0216) ✅                褔 (0.0378)   
7               Lincoln (0.0207)         phishing (0.0334)   
8            Vaccines (0.0137) ✅                🙏 (0.0229)   
9                Rivers (0.0106)        Daughters (0.0220)   
10                Vegan (0.0102)           Excise (0.0187)   
11          Allergy (8.65e-03) ✅       Vaccines (0.0139) ✅   
12   Bioinformatics (8.63e-03) ✅             json (0.0134)   
13         helpline (7.91e-03) ✅        photoshop (0.0123)   
14                  🙏 (7.30e-03)          Lincoln (0.0123)   
15                 財布 (6.73e-03)   Bioinformatics (0.0118)   
16                dmg (5.70e-03)                氪 (0.0108)   
17         douleurs (5.45e-03) ✅          Vegan (0.0108) ✅   
18          instagram (5.03e-03)       douleurs (0.0104) ✅   
19              oauth (4.81e-03)     Calories (8.44e-03) ✅   

                                                layer_23  \
                        pos_3                     pos_-3   
0        bollywood (0.5234) ✅                вы (0.1299)   
1                 交換 (0.1318)            CUDA (0.0614) ✅   
2                  褔 (0.0625)                ્ર (0.0477)   
3        photoshop (0.0334) ✅              '.', (0.0422)   
4                  ॕ (0.0334)             Ext (0.0226) ✅   
5           douleurs (0.0229)               ]', (0.0208)   
6           Movies (0.0157) ✅       Isometric (0.0176) ✅   
7               xray (0.0123)              idor (0.0176)   
8                dmg (0.0109)             січня (0.0176)   
9         Vaccines (6.59e-03)             سلسلہ (0.0155)   
10        படத்தின் (5.80e-03)             anine (0.0137)   
11             नही (5.13e-03)              ग्लो (0.0131)   
12         Lincoln (5.13e-03)              کردم (0.0121)   
13        Calories (4.52e-03)              }"), (0.0121)   
1